<a id="sec-3"></a>
<div style="background-color:#4EA72E; padding:20px; border-radius:10px; border-left:8px solid #2E6B1A;">
  <h3 style="margin:0; color:#FFFFFF;"><b>3. Advanced Topic - Interactive Maps</b></h3>
</div>

In this section we build interactive world maps that show where the most streamed artists come from. To do this, we first need the birthplace or origin of each artist and the coordinates of that place, which are not in our dataset. Both are collected from Wikipedia through web scraping, starting from the Wikipedia Main Page.

<a id="sec-3-1"></a>
<div style="background-color:#F1FAEE; padding:20px; border-radius:10px; border-left:8px solid #4EA72E; color:#2E6B1A;">
  <b>3.1. Web Scraping</b>
</div>

Wikipedia pages are static HTML, so we use `requests` to download each page and `BeautifulSoup` to read it. We don't need Selenium here, since there are no buttons, cookies or JavaScript content to deal with, and `requests` is much faster when we have to visit hundreds of pages.

The process for each artist is the following:
1. Search for the artist using the search box of the Wikipedia Main Page.
2. Check if the page found is really about a musician and if the place matches the artist's country in our dataset.
3. Read the birthplace (solo artists) or origin (groups) from the information box on the right side of the page.
4. Open the page of that place and read its coordinates.

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import re
import os

<h3 style="color:#4EA72E;"><b>BEST PRACTICES</b></h3>

Before scraping, we check Wikipedia's `robots.txt`, which says which parts of the website automated programs can visit. Article pages (`/wiki/...`) are allowed. We also add a pause between requests so we don't overload the website, and we identify our program with a `User-Agent`, as Wikipedia asks scripts to do.

In [ ]:
headers = {'User-Agent': 'PDS-NOVA-IMS-student-project/1.0 (educational use)'}

robots = requests.get('https://en.wikipedia.org/robots.txt', headers=headers, timeout=10)
print(f"Status code: {robots.status_code}")

<h3 style="color:#4EA72E;"><b>STARTING FROM THE MAIN PAGE</b></h3>

We open the Wikipedia Main Page and look for its search box in the HTML. The search box is a `<form>` with `id="searchform"`: its `action` attribute gives the address used for searches, and its hidden `<input>` fields are the extra values sent with each search.

In [ ]:
main_url = 'https://en.wikipedia.org/wiki/Main_Page'
response = requests.get(main_url, headers=headers, timeout=10)
print(f"Status code: {response.status_code}")

main_soup = BeautifulSoup(response.text, 'html.parser')
search_form = main_soup.find('form', {'id': 'searchform'})

# Address used by the search box and hidden values sent with it
search_url = 'https://en.wikipedia.org' + search_form.get('action')
search_params = {field.get('name'): field.get('value') for field in search_form.find_all('input', {'type': 'hidden'})}

print(f"Search URL: {search_url}")
print(f"Hidden fields: {search_params}")

When we search for a name that is the exact title of an article, Wikipedia goes directly to that article, just like pressing Enter in the search box. The function below does that search and returns the page already parsed with BeautifulSoup.

In [ ]:
def search_wikipedia(query):
    params = search_params.copy()
    params['search'] = query
    params['go'] = 'Go'  # same as pressing Enter in the search box
    response = requests.get(search_url, params=params, headers=headers, timeout=10)
    time.sleep(0.3)  # pause between requests
    if response.status_code != 200:
        return None
    return BeautifulSoup(response.text, 'html.parser')

<h3 style="color:#4EA72E;"><b>READING THE INFORMATION BOX</b></h3>

The artist information is in the box on the right side of the page (the infobox), which is a `<table>` with class `infobox`. Each row has a label in a `<th>` (for example *Born*, *Origin* or *Genres*) and the value in the next `<td>`.

A page is accepted as a musician page if its infobox has a *Genres* or *Years active* row. This avoids pages about other people or things with the same name.

In [ ]:
def infobox_value(soup, label):
    infobox = soup.find('table', class_='infobox')
    if infobox is None:
        return None
    for th in infobox.find_all('th', class_='infobox-label'):
        if th.text.strip() == label:
            return th.find_next_sibling('td')
    return None


def is_musician(soup):
    return infobox_value(soup, 'Genres') is not None or infobox_value(soup, 'Years active') is not None

For solo artists we look for the place of birth, inside the *Born* row. On most pages it is in a tag with class `birthplace`, and when it isn't, it is the last line of the cell, after the date. For groups we use the *Origin* row. If the main option is not available, we try the other one.

In [ ]:
def clean_text(tag):
    for ref in tag.find_all('sup'):  # remove references like [1]
        ref.decompose()
    text = tag.get_text(' ', strip=True)
    return text.replace(' ,', ',').strip(' ,')


def get_place(soup, artist_type):
    born = infobox_value(soup, 'Born')
    origin = infobox_value(soup, 'Origin')

    if artist_type == 'Solo':
        options = [('Born', born), ('Origin', origin)]
    else:
        options = [('Origin', origin), ('Born', born)]

    for source, cell in options:
        if cell is None:
            continue
        if source == 'Origin':
            return cell, clean_text(cell), source
        tag = cell.find(class_='birthplace')
        if tag is None:
            # place is the last line of the cell, after the date of birth
            last_line = re.split(r'<br\s*/?>', str(cell))[-1]
            tag = BeautifulSoup(last_line, 'html.parser')
        text = clean_text(tag)
        if text != '' and '(age' not in text and not text[-4:].isdigit():
            return tag, text, source
    return None, None, None

<h3 style="color:#4EA72E;"><b>MATCHING THE RIGHT ARTIST</b></h3>

Many artists share their name with other people, bands or things (for example *Drake* or *Queen*). Wikipedia separates them by adding a description to the title, such as *Drake (musician)* or *Queen (band)*. So, if the first page found is not a musician, we try these variations.

To make sure the artist is correctly matched, we also compare the place found with the `Country of Origin` in our dataset. Since Wikipedia writes some countries in different ways (for example *U.S.* or *England*), we list the alternative names. Older articles sometimes only write the US state, so the states are also accepted for the United States.

In [ ]:
country_names = {
    'United States': ['United States', 'U.S.', 'US', 'USA', 'Puerto Rico'],
    'United Kingdom': ['United Kingdom', 'UK', 'U.K.', 'England', 'Scotland', 'Wales', 'Northern Ireland'],
    'South Korea': ['South Korea', 'Korea'],
    'DR Congo': ['Congo', 'Zaire'],
    'Russia': ['Russia', 'Soviet Union'],
}

us_states = ['Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Colorado', 'Connecticut', 'Delaware', 'Florida',
             'Georgia', 'Hawaii', 'Idaho', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine',
             'Maryland', 'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi', 'Missouri', 'Montana', 'Nebraska',
             'Nevada', 'New Hampshire', 'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota', 'Ohio',
             'Oklahoma', 'Oregon', 'Pennsylvania', 'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee', 'Texas',
             'Utah', 'Vermont', 'Virginia', 'Washington', 'West Virginia', 'Wisconsin', 'Wyoming']


def country_matches(place, country):
    names = country_names.get(country, [country])
    if country == 'United States':
        names = names + us_states
    for name in names:
        # the name must appear as a whole word (so 'US' does not match 'Houston')
        if re.search(r'(?<!\w)' + re.escape(name) + r'(?!\w)', place):
            return True
    return False

In [ ]:
variations = ['', ' (musician)', ' (singer)', ' (rapper)', ' (band)', ' (group)', ' (DJ)',
              ' (producer)', ' (record producer)', ' (singer-songwriter)', ' (composer)', ' (lyricist)']


def find_artist(name, artist_type, country):
    first_found = None  # first musician page found, kept if no page matches the country
    for variation in variations:
        soup = search_wikipedia(name + variation)
        if soup is None or not is_musician(soup):
            continue
        tag, place, source = get_place(soup, artist_type)
        if place is None:
            continue
        page = soup.find('link', {'rel': 'canonical'}).get('href')
        result = {'Wikipedia Page': page, 'Place': place, 'Place Source': source, 'tag': tag}
        if country_matches(place, country):
            result['Country Check'] = True
            return result
        if first_found is None:
            result['Country Check'] = False
            first_found = result
    return first_found

<h3 style="color:#4EA72E;"><b>GETTING THE COORDINATES</b></h3>

The place in the infobox usually has links to the city, region and country, from the most specific to the least specific. We open the first link and read the coordinates, which appear at the top of the page inside a `<span>` with class `geo` (for example `43.74167; -79.37333`). If that page has no coordinates, we try the next link. If none of the links work, we search the name of the place, removing its first part each time (for example *St. Mary's Hospital, Toronto, Canada* → *Toronto, Canada*).

Many artists come from the same cities, so we save the coordinates of each place we visit in a dictionary and don't download the same page twice.

In [ ]:
coordinates_found = {}


def read_coordinates(soup):
    if soup is None:
        return None
    geo = soup.find('span', class_='geo')
    if geo is None:
        return None
    lat, lon = geo.text.split(';')
    return float(lat), float(lon)


def get_coordinates(tag, place):
    # 1) follow the links inside the place
    for link in tag.find_all('a'):
        href = link.get('href', '')
        if not href.startswith('/wiki/') or ':' in href:  # skip files, references and special pages
            continue
        if href not in coordinates_found:
            response = requests.get('https://en.wikipedia.org' + href, headers=headers, timeout=10)
            time.sleep(0.3)
            coordinates_found[href] = read_coordinates(BeautifulSoup(response.text, 'html.parser'))
        if coordinates_found[href] is not None:
            return coordinates_found[href]

    # 2) search the place name, removing the first part each time
    parts = [p.strip() for p in place.split(',')]
    for i in range(len(parts)):
        query = ', '.join(parts[i:])
        if query not in coordinates_found:
            coordinates_found[query] = read_coordinates(search_wikipedia(query))
        if coordinates_found[query] is not None:
            return coordinates_found[query]
    return None, None

<h3 style="color:#4EA72E;"><b>TESTING WITH ONE ARTIST</b></h3>

Before running for all artists, we test the process with Drake. Searching only "Drake" leads to a page that is not about the musician, so this also tests the title variations.

In [ ]:
drake = find_artist('Drake', 'Solo', 'Canada')
print(f"Page: {drake['Wikipedia Page']}")
print(f"Place: {drake['Place']} ({drake['Place Source']})")
print(f"Country check: {drake['Country Check']}")
print(f"Coordinates: {get_coordinates(drake['tag'], drake['Place'])}")

<h3 style="color:#4EA72E;"><b>SCRAPING ALL ARTISTS</b></h3>

We now repeat the process for every artist. This takes a while (around 15 to 25 minutes), so the results are saved in `artist_locations.csv`. If this file already exists, it is loaded instead of scraping everything again. If a page gives an error, we print the artist's name and move on, so one problem does not stop the whole process.

In [ ]:
locations_file = 'artist_locations.csv'

if os.path.exists(locations_file):
    locations = pd.read_csv(locations_file, index_col=0)
    print(f"Loaded {len(locations)} artists from {locations_file}")
else:
    rows = []
    start = time.time()
    for n, (artist_id, artist) in enumerate(artist_df.iterrows()):
        row = {'Artist ID': artist_id, 'Artist Name': artist['Artist Name']}
        try:
            result = find_artist(artist['Artist Name'], artist['Artist Type'], artist['Country of Origin'])
            if result is not None:
                lat, lon = get_coordinates(result['tag'], result['Place'])
                row['Wikipedia Page'] = result['Wikipedia Page']
                row['Place'] = result['Place']
                row['Place Source'] = result['Place Source']
                row['Country Check'] = result['Country Check']
                row['Latitude'] = lat
                row['Longitude'] = lon
        except Exception as error:
            print(f"Error with {artist['Artist Name']}: {error}")
        rows.append(row)
        if n % 50 == 0:
            print(f"{n}/{len(artist_df)} artists ({time.time() - start:.0f} seconds)")

    locations = pd.DataFrame(rows).set_index('Artist ID')
    locations.to_csv(locations_file)
    print(f"Finished in {(time.time() - start) / 60:.1f} minutes")

locations.head()

<h3 style="color:#4EA72E;"><b>CHECKING THE RESULTS</b></h3>

In [ ]:
print(f"Artists: {len(locations)}")
print(f"Musician page and place found: {locations['Place'].notna().sum()}")
print(f"Coordinates found: {locations['Latitude'].notna().sum()}")
print(f"Place matches the country in the dataset: {(locations['Country Check'] == True).sum()}")
print()
print(locations['Place Source'].value_counts())

Artists for which no musician page was found on Wikipedia:

In [ ]:
locations[locations['Place'].isna()]['Artist Name']

Artists whose place does not match the country in our dataset. These were checked one by one: a mismatch is not always an error, since an artist can be born in one country and start their career in another.

In [ ]:
mismatches = locations[locations['Country Check'] == False].join(artist_df['Country of Origin'])
mismatches[['Artist Name', 'Place', 'Country of Origin', 'Wikipedia Page']]

When the page found belongs to a different person or band, we write the correct Wikipedia title in the dictionary below and that artist is scraped again, still only using Wikipedia.

In [ ]:
# Artist Name -> correct Wikipedia title (filled in after checking the table above)
correct_titles = {}

for name, title in correct_titles.items():
    artist_id = artist_df[artist_df['Artist Name'] == name].index[0]
    soup = search_wikipedia(title)
    tag, place, source = get_place(soup, artist_df.loc[artist_id, 'Artist Type'])
    if place is not None:
        lat, lon = get_coordinates(tag, place)
        locations.loc[artist_id, ['Wikipedia Page', 'Place', 'Place Source', 'Country Check', 'Latitude', 'Longitude']] = \
            ['https://en.wikipedia.org/wiki/' + title.replace(' ', '_'), place, source, True, lat, lon]

if len(correct_titles) > 0:
    locations.to_csv(locations_file)

<a id="sec-3-2"></a>
<div style="background-color:#F1FAEE; padding:20px; border-radius:10px; border-left:8px solid #4EA72E; color:#2E6B1A;">
  <b>3.2. Interactive Maps</b>
</div>

We join the coordinates with our dataset and calculate the percentage of solo and collaborative streams of each artist, which are shown when hovering over the markers.

Many artists come from the same city (for example Los Angeles, Atlanta or London), so their markers would be exactly on top of each other and only one would be visible. To avoid this, we move each marker by a very small random distance (around 10 km), only for the map.

In [ ]:
map_df = artist_df.join(locations[['Place', 'Latitude', 'Longitude']]).dropna(subset=['Latitude', 'Longitude'])

map_df['% Solo'] = map_df['Solo Streams'] / map_df['Total Streams'] * 100
map_df['% Collaborative'] = map_df['Collaborative Streams'] / map_df['Total Streams'] * 100

np.random.seed(42)  # same positions every time the notebook is run
map_df['Map Latitude'] = map_df['Latitude'] + np.random.uniform(-0.1, 0.1, len(map_df))
map_df['Map Longitude'] = map_df['Longitude'] + np.random.uniform(-0.1, 0.1, len(map_df))

print(f"Artists on the map: {len(map_df)} of {len(artist_df)}")

<h3 style="color:#4EA72E;"><b>MAP 1 - ARTISTS AROUND THE WORLD</b></h3>

Each marker is an artist, sized by their total number of streams. Hovering over a marker shows the artist's name, birthplace or origin, total streams, percentage of solo and collaborative streams, and also their genre and language.

In [ ]:
fig = px.scatter_geo(map_df, lat='Map Latitude', lon='Map Longitude', size='Total Streams', hover_name='Artist Name',
                     custom_data=['Place', 'Total Streams', '% Solo', '% Collaborative', 'Primary Genre', 'Primary Language'],
                     projection='natural earth', size_max=35, color_discrete_sequence=['#4EA72E'])

fig.update_traces(marker=dict(opacity=0.7, line=dict(width=0.5, color='#2E6B1A')),
                  hovertemplate='<b>%{hovertext}</b><br><br>'
                                'Birthplace/Origin: %{customdata[0]}<br>'
                                'Total Streams: %{customdata[1]:,.1f}M<br>'
                                '% Solo Streams: %{customdata[2]:.2f}%<br>'
                                '% Collaborative Streams: %{customdata[3]:.2f}%<br>'
                                'Genre: %{customdata[4]} | Language: %{customdata[5]}<extra></extra>')

fig.update_geos(showcountries=True, countrycolor='white', landcolor='#E8EEE6', showocean=True, oceancolor='#F7FBF5')
fig.update_layout(title='Where do the most streamed artists come from?', height=650, margin=dict(l=0, r=0, t=50, b=0))
fig.show()

<h3 style="color:#4EA72E;"><b>MAP 2 - DENSITY OF STREAMS</b></h3>

In this map, each artist adds their total streams to the area of their birthplace or origin, so the darker areas are the places that generate more streams, not only the ones with more artists.

In [ ]:
fig = px.density_map(map_df, lat='Latitude', lon='Longitude', z='Total Streams', radius=20,
                     hover_name='Artist Name', hover_data={'Place': True, 'Total Streams': ':,.1f', 'Latitude': False, 'Longitude': False},
                     center=dict(lat=25, lon=0), zoom=0.8, map_style='carto-positron', color_continuous_scale='Greens')

fig.update_layout(title='Global distribution of streams by birthplace or origin', height=600,
                  coloraxis_colorbar=dict(title='Total Streams (M)'), margin=dict(l=0, r=0, t=50, b=0))
fig.show()

*(Conclusion to write after running the maps: which regions concentrate more artists and streams, for example North America and the UK, and which other regions stand out, such as Latin America, South Korea or India.)*